Copied from part1-2

**BRONZE**

In [0]:
from pyspark.sql.functions import current_timestamp, date_format

#if want to test code else where change these vars for yourseld
catalog_name = "26100355-pa2" 
schema_name = "bronze"
volume_name = "temp"
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
#made vol in bronze layer for me
landing_zone = f"{base_path}/flight_landing_zone"
schema_location = f"{base_path}/schemas/bronze_flights"
checkpoint_path = f"{base_path}/checkpoints/bronze_flights"

print(f"Copying raw data to {landing_zone}...")
dbutils.fs.cp("dbfs:/databricks-datasets/flights/departuredelays.csv", landing_zone)
print("Start Auto-Loader read st")
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true") 
    .load(landing_zone))
bronze_stream = raw_stream.withColumn(
    "ingestion_timestamp",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)


print(f"Writing stream to {catalog_name}.{schema_name}.flights...")
write_query = (bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`{schema_name}`.`flights`"))
write_query.awaitTermination()
print("Bronze done")

Copying raw data to /Volumes/26100355-pa2/bronze/temp/flight_landing_zone...
Start Auto-Loader read st
Writing stream to 26100355-pa2.bronze.flights...
Bronze done


**SILVER**

In [0]:
from pyspark.sql.functions import col, concat, lit, to_timestamp, date_format

catalog_name = "26100355-pa2"
bronze_table = f"`{catalog_name}`.`bronze`.`flights`"

print("Reading from Bronze stream...")
bronze_stream = spark.readStream.table(bronze_table)

silver_df = (bronze_stream
    .withColumn("delay", col("delay").cast("integer"))
    .withColumn("distance", col("distance").cast("integer"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("raw_date_with_year", concat(lit("2025"), col("date")))
    .withColumn("event_timestamp", to_timestamp(col("raw_date_with_year"), "yyyyMMddHHmm"))
    .withColumn("date", date_format(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .drop("raw_date_with_year")
)

Reading from Bronze stream...


In [0]:
from pyspark.sql.functions import col, concat, lit, to_timestamp, date_format, when, array, array_sort, concat_ws, broadcast

catalog_name = "26100355-pa2"
bronze_table = f"`{catalog_name}`.`bronze`.`flights`"
silver_table = f"`{catalog_name}`.`silver`.`flights`"
checkpoint_path = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/silver_flights"

print("Processing Silver Layer...")

bronze_stream = spark.readStream.table(bronze_table)

silver_part1_df = (bronze_stream
    .withColumn("delay", col("delay").cast("integer"))
    .withColumn("distance", col("distance").cast("integer"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("event_timestamp", to_timestamp(concat(lit("2025"), col("date")), "yyyyMMddHHmm"))
    .withColumn("date", date_format(col("event_timestamp"), "yyyy-MM-dd HH:mm:ss"))
)

ref_df = spark.read.csv("/databricks-datasets/flights/airport-codes-na.txt", header=True, sep="\t")

org_ref = broadcast(ref_df).alias("org")
dst_ref = broadcast(ref_df).alias("dst")

silver_part2_df = (silver_part1_df
    .join(org_ref, col("origin") == col("org.IATA"), "left")
    .join(dst_ref, col("destination") == col("dst.IATA"), "left")
    .withColumn("departure", concat_ws(", ", col("org.City"), col("org.State")))
    .withColumn("arrival", concat_ws(", ", col("dst.City"), col("dst.State")))
    .filter((col("departure") != "") & (col("arrival") != ""))
    .withColumn("take_off_status", 
        when(col("delay") < 0, "Early")
        .when((col("delay") >= 0) & (col("delay") < 10), "On Time")
        .when((col("delay") >= 10) & (col("delay") < 30), "Late")
        .when(col("delay") >= 30, "Delay")
    )
    .withColumn("standardized_route", concat_ws("-", array_sort(array(col("origin"), col("destination")))))
    .withColumn("seq", lit(3))
    .select(
        "date", "departure", "arrival", "standardized_route", 
        "distance", "delay", "take_off_status", 
        "ingestion_timestamp", "event_timestamp", "seq"
    )
)

write_query = (silver_part2_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

write_query.awaitTermination()
print("Silver Layer complete.")

Processing Silver Layer...
Silver Layer complete.


In [0]:
from pyspark.sql.functions import col, from_json, row_number
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

catalog_name = "26100355-pa2"
silver_table_name = f"`{catalog_name}`.`silver`.`flights`"
cdc_checkpoint = f"/Volumes/{catalog_name}/bronze/temp/checkpoints/cdc_silver"
cdc_zone_part2 = f"/Volumes/{catalog_name}/bronze/temp/cdc_zone_part2"

payload_schema = StructType([
    StructField("date", StringType(), True),
    StructField("departure", StringType(), True),
    StructField("arrival", StringType(), True),
    StructField("standardized_route", StringType(), True),
    StructField("distance", IntegerType(), True),
    StructField("delay", IntegerType(), True),
    StructField("take_off_status", StringType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

def merge_cdc_to_silver(microBatchDF, batchId):
    parsed_df = microBatchDF.withColumn("data", from_json(col("payload"), payload_schema)).select("data.*", "action", "seq")
    
    window_spec = Window.partitionBy("date", "standardized_route").orderBy(col("seq").desc())
    
    deduped_df = parsed_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")
    
    target_table = DeltaTable.forName(spark, silver_table_name)
    
    target_table.alias("target").merge(
        deduped_df.alias("source"),
        "target.date = source.date AND target.standardized_route = source.standardized_route"
    ).whenMatchedUpdate(
        condition="(source.action = 'U' OR source.action = 'update') AND source.seq > target.seq",
        set={
            "delay": "source.delay",
            "take_off_status": "source.take_off_status",
            "seq": "source.seq"
        }
    ).whenMatchedDelete(
        condition="(source.action = 'D' OR source.action = 'delete') AND source.seq > target.seq"
    ).execute()

cdc_schema_location = f"/Volumes/{catalog_name}/bronze/temp/schemas/cdc_silver"
dbutils.fs.mkdirs(cdc_zone_part2)

cdc_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", cdc_schema_location)
    .load(cdc_zone_part2)
)

cdc_csv_schema = StructType([
    StructField("payload", StringType(), True),
    StructField("action", StringType(), True),
    StructField("seq", IntegerType(), True)
])

cdc_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(cdc_csv_schema)
    .load(cdc_zone_part2)
)
cdc_query = cdc_stream.writeStream.foreachBatch(merge_cdc_to_silver).option("checkpointLocation", cdc_checkpoint).trigger(availableNow=True).start()

cdc_query.awaitTermination()

**GOLD**

In [0]:
from pyspark.sql.functions import col, window, avg, sum, expr, round, date_format, count

catalog_name = "26100355-pa2"
silver_table = f"`{catalog_name}`.`silver`.`flights`"
base_checkpoint = f"/Volumes/{catalog_name}/bronze/temp/checkpoints"

silver_stream = spark.readStream.table(silver_table).withWatermark("event_timestamp", "7 days")

req1_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "30 days"))
    .agg(round(avg(col("delay") / 60), 2).alias("moving_avg_delay"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "moving_avg_delay"
    )
)

req1_query = (req1_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req1_monthly_delays")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`monthly_delays`")
)

req2_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "14 days"))
    .agg(round(sum(col("distance")), 0).alias("total_distance"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "total_distance"
    )
)

req2_query = (req2_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req2_biweekly_distance")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`biweekly_distance`")
)

req3_df = (silver_stream
    .withColumn("delay_hours", col("delay") / 60)
    .groupBy(window(col("event_timestamp"), "30 days"))
    .agg(
        expr("max_by(standardized_route, delay_hours)").alias("worst_route"),
        round(expr("max(delay_hours)"), 2).alias("max_delay")
    )
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "worst_route",
        "max_delay"
    )
)

req3_query = (req3_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/req3_monthly_max_delay")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`monthly_max_delay`")
)

task4_df = (silver_stream
    .groupBy(window(col("event_timestamp"), "1 hour"))
    .agg(count("*").alias("flight_count"))
    .select(
        date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        "flight_count"
    )
)

task4_query = (task4_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{base_checkpoint}/task4_hourly_counts")
    .trigger(availableNow=True)
    .toTable(f"`{catalog_name}`.`gold`.`hourly_flight_counts`")
)

print("Executing Gold Layer streams...")
req1_query.awaitTermination()
req2_query.awaitTermination()
req3_query.awaitTermination()
task4_query.awaitTermination()
print("Gold Layer aggregations complete.")

Executing Gold Layer streams...
Gold Layer aggregations complete.


In [0]:
print("Requirement 1")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`monthly_delays` ORDER BY window_start LIMIT 5"))

print("Requirement 2")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`biweekly_distance` ORDER BY window_start LIMIT 5"))

print("Requirement 3")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`monthly_max_delay` ORDER BY window_start LIMIT 5"))

print("Req 4:")
display(spark.sql(f"SELECT * FROM `{catalog_name}`.`gold`.`hourly_flight_counts` ORDER BY window_start LIMIT 5"))

Requirement 1


window_start,window_end,moving_avg_delay
2024-12-13 00:00:00,2025-01-12 00:00:00,0.44
2025-01-12 00:00:00,2025-02-11 00:00:00,0.17
2025-02-11 00:00:00,2025-03-13 00:00:00,0.18
2025-03-13 00:00:00,2025-04-12 00:00:00,0.15


Requirement 2


window_start,window_end,total_distance
2024-12-19 00:00:00,2025-01-02 00:00:00,9967392
2025-01-02 00:00:00,2025-01-16 00:00:00,143764431
2025-01-16 00:00:00,2025-01-30 00:00:00,137124712
2025-01-30 00:00:00,2025-02-13 00:00:00,137128874
2025-02-13 00:00:00,2025-02-27 00:00:00,145578616


Requirement 3


window_start,window_end,worst_route,max_delay
2024-12-13 00:00:00,2025-01-12 00:00:00,CMH-LAX,24.35
2025-01-12 00:00:00,2025-02-11 00:00:00,DFW-FLL,27.27
2025-02-11 00:00:00,2025-03-13 00:00:00,DFW-TPA,27.37
2025-03-13 00:00:00,2025-04-12 00:00:00,DFW-ELP,24.5


Req 4:


window_start,window_end,flight_count
2025-01-01 00:00:00,2025-01-01 01:00:00,20
2025-01-01 01:00:00,2025-01-01 02:00:00,8
2025-01-01 02:00:00,2025-01-01 03:00:00,3
2025-01-01 05:00:00,2025-01-01 06:00:00,136
2025-01-01 06:00:00,2025-01-01 07:00:00,634
